# I Spent a Week With LangChain. Here's What Nobody Tells You.
### Deep Technical Blog — LangChain

**Author:** Abhishek Kharat  
*Medium Blog:* *(https://medium.com/p/d079b6bbf2a4?postPublishedType=initial)*  
*GitHub:* https://github.com/AbhishekKharat04/Innomatics-Agentic-AI-Internship-2026/tree/main/GenAI%20–%20Prompt
---

### Stack used (all FREE — no paid API keys needed)
| Component | Library | Cost |
|---|---|---|
| LLM | Groq (Llama 3.1 8B) | Free tier |
| Embeddings | HuggingFace sentence-transformers | Free |
| Vector Store | ChromaDB (local) | Free |
| Framework | LangChain Core | Open source |

---

## 📦 Step 0: Install Dependencies
> Run this cell first. Groq is free — sign up at https://console.groq.com and get your API key.

In [1]:
!pip install -U langchain langchain-community langchain-core langchain-groq --quiet

In [2]:
!pip install langchain-core langchain-groq langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers --quiet
print("✅ All packages installed!")

✅ All packages installed!


## 🔑 Step 1: Set Up Groq API Key
Get your FREE key at: https://console.groq.com → No credit card needed.

In [3]:
import os
from getpass import getpass

# Paste your Groq API key when prompted — it won't be saved or shown
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")
print("✅ API key set!")

Enter your Groq API Key: ··········
✅ API key set!


---
## Component 1: LLMs and Chat Models

LangChain wraps different model providers (OpenAI, Groq, Anthropic, HuggingFace) behind one consistent interface.
Swap the provider, keep your pipeline code identical.

Here we use **Groq's free Llama 3.1** — same LangChain interface, zero cost.

In [4]:
from langchain_groq import ChatGroq

# Initialize the LLM — temperature=0 for deterministic outputs
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=600,
)

# Basic LLM call — text in, text out
response = llm.invoke("In one sentence, what problem does LangChain solve?")
print("📌 Basic LLM call output:")
print(response.content)

📌 Basic LLM call output:
LangChain is an open-source Python library that solves the problem of integrating and orchestrating multiple AI models and data sources to create more powerful and flexible AI applications.


---
## Component 2: Prompt Templates

Instead of building prompts with f-strings, `PromptTemplate` and `ChatPromptTemplate`
separate your template structure from your application logic.
Define once → reuse everywhere → change the template without touching the rest of your code.

In [5]:
from langchain_core.prompts import PromptTemplate

# Single-variable template — completely reusable
explain_template = PromptTemplate(
    input_variables=["concept"],
    template="Explain {concept} in 2-3 sentences. Use a real-world analogy."
)

# Same template — different outputs
concepts = ["vector embeddings", "LangChain chains", "RAG (Retrieval-Augmented Generation)"]

print("📌 PromptTemplate reusability — same template, 3 different prompts:")
print("-" * 60)
for c in concepts:
    print(explain_template.format(concept=c))
print("-" * 60)

📌 PromptTemplate reusability — same template, 3 different prompts:
------------------------------------------------------------
Explain vector embeddings in 2-3 sentences. Use a real-world analogy.
Explain LangChain chains in 2-3 sentences. Use a real-world analogy.
Explain RAG (Retrieval-Augmented Generation) in 2-3 sentences. Use a real-world analogy.
------------------------------------------------------------


In [6]:
from langchain_core.prompts import ChatPromptTemplate

# Chat template with system behavior + dynamic user input
# This is what you use for role-based, conversational prompts
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Keep your response under 4 sentences. Be direct."),
    ("human", "Explain {topic} to me.")
])

# Format and inspect the messages — see the structure
messages = chat_template.format_messages(role="senior ML engineer", topic="attention mechanisms")

print("📌 ChatPromptTemplate — formatted messages:")
for msg in messages:
    print(f"[{msg.type.upper()}]: {msg.content}")

📌 ChatPromptTemplate — formatted messages:
[SYSTEM]: You are a senior ML engineer. Keep your response under 4 sentences. Be direct.
[HUMAN]: Explain attention mechanisms to me.


---
## Component 3: Chains (LCEL — LangChain Expression Language)

The `|` pipe operator connects components into a pipeline.
`prompt | model | parser` is the basic pattern.
Each step is independent — swap any piece without touching the rest.

In [7]:
from langchain_core.output_parsers import StrOutputParser

# Build the chain using LCEL pipe operator
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Answer clearly and concisely."),
    ("human", "Answer this in 3 bullet points: {question}")
])

# Chain: prompt → LLM → parse output to plain string
qa_chain = qa_prompt | llm | StrOutputParser()

# Invoke — one call runs all 3 steps
result = qa_chain.invoke({"question": "Why are prompt templates better than f-strings in LLM applications?"})

print("📌 Simple Chain output:")
print(result)

📌 Simple Chain output:
Here are three reasons why prompt templates are better than f-strings in LLM (Large Language Model) applications:

• **Flexibility and Customizability**: Prompt templates allow for more flexibility and customizability in generating text. They can be easily modified to accommodate different input formats, styles, and requirements, making them more adaptable to various LLM applications.

• **Improved Interpretability and Control**: Prompt templates provide more control over the output generation process, allowing developers to specify the exact structure and format of the generated text. This improves interpretability and makes it easier to debug and fine-tune the model.

• **Better Handling of Complex Scenarios**: Prompt templates can handle complex scenarios and edge cases more effectively than f-strings. They can be designed to accommodate multiple input variables, conditional statements, and nested structures, making them more suitable for generating text in co

In [8]:
# Demonstrating chain reusability — same chain, different inputs
questions = [
    "What is the difference between a chain and an agent in LangChain?",
    "When would you use ConversationSummaryMemory instead of ConversationBufferMemory?",
]

print("📌 Chain reusability — 2 questions, same chain:")
for i, q in enumerate(questions, 1):
    print(f"\n--- Question {i}: {q}")
    print(qa_chain.invoke({"question": q}))

📌 Chain reusability — 2 questions, same chain:

--- Question 1: What is the difference between a chain and an agent in LangChain?
Here are 3 key differences between a chain and an agent in LangChain:

* **Purpose**: A chain in LangChain is a sequence of tasks or operations that are executed in a specific order, whereas an agent is a more complex entity that can perform multiple tasks, make decisions, and interact with the environment.
* **Structure**: A chain is typically a linear sequence of tasks, whereas an agent can have a more complex structure, including multiple components, such as a task executor, a decision maker, and a memory.
* **Behavior**: A chain executes tasks in a deterministic way, following a predefined sequence, whereas an agent can exhibit more complex behavior, such as learning, adapting, and making decisions based on its environment and goals.

--- Question 2: When would you use ConversationSummaryMemory instead of ConversationBufferMemory?
Here are three key poin

---
## Component 4: Memory

LLMs are stateless — each call forgets everything from before.
Memory fixes this by injecting conversation history back into each new prompt automatically.

In [9]:
# Modern approach — langchain.memory is deprecated in newer versions
# We manually manage chat history using ChatPromptTemplate + a list
# This is the LCEL-native way and works with all current LangChain versions

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# Prompt that accepts a history list — MessagesPlaceholder injects it
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Remember everything the user tells you."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

memory_chain = memory_prompt | llm | StrOutputParser()

# Manually maintain the chat history list
chat_history = []

def chat(user_input: str) -> str:
    """Send a message and automatically update history."""
    response = memory_chain.invoke({
        "history": chat_history,
        "input": user_input
    })
    # Append both turns to history for next call
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    return response

# Turn 1 — introduce context
r1 = chat("My name is Abhishek and I'm building a document Q&A system.")
print("Turn 1:", r1)

# Turn 2 — model should remember the context from turn 1
r2 = chat("What would you recommend as the most important component for my use case?")
print("\nTurn 2:", r2)

# Turn 3 — test if it remembers the name
r3 = chat("What's my name and what am I building?")
print("\nTurn 3:", r3)

# Show what's stored in history
print(f"\n📌 History has {len(chat_history)} messages stored across {len(chat_history)//2} turns")

Turn 1: Nice to meet you, Abhishek. I'd be happy to help you with your document Q&A system. I can remember everything you tell me, so feel free to share your ideas, requirements, and progress. What specific aspects of the system are you working on or would like to discuss?

Turn 2: For a document Q&A system, I would recommend a robust **Information Retrieval (IR) component** as the most important component. This component will be responsible for searching and retrieving relevant information from the documents to answer user queries.

A good IR component should be able to:

1. Index the documents: Create a searchable index of the documents to enable fast and efficient querying.
2. Handle natural language queries: Understand and process user queries in a natural language format.
3. Retrieve relevant documents: Return a list of relevant documents that match the user's query.
4. Rank documents: Rank the retrieved documents based on their relevance to the query.

Some popular IR techniques 

---
## Component 5: Tools

Tools are Python functions that an agent can call when it needs to do something
the LLM isn't reliable at — exact calculations, API calls, database queries.

The `@tool` decorator is all you need to expose a function to an agent.

In [10]:
from langchain_core.tools import tool

# Tool 1: Exact calculation (LLMs hallucinate these — tools don't)
@tool
def calculate_percentage(value: float, percentage: float) -> str:
    """Calculate what percentage of a given value is. E.g. 15% of 200."""
    result = (percentage / 100) * value
    return f"{percentage}% of {value} = {result}"

# Tool 2: Word count
@tool
def count_words(text: str) -> str:
    """Counts the number of words in a given text string."""
    count = len(text.split())
    return f"The text contains {count} words."

# Test tools independently — always do this before wiring into an agent
print("📌 Tool tests:")
print(calculate_percentage.invoke({"value": 4500, "percentage": 18}))
print(count_words.invoke({"text": "LangChain makes building LLM applications much easier and more modular"}))

📌 Tool tests:
18.0% of 4500.0 = 810.0
The text contains 10 words.


---
## Component 6: Agents

An agent is a chain that can decide for itself what to do next.
It follows the ReAct loop: Think → Act (pick a tool) → Observe result → Repeat until done.

Unlike a fixed chain where every step is predetermined,
the agent reasons about which tool to call — or whether to call one at all.

In [17]:
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# Re-define tools
@tool
def calculate_percentage(value: float, percentage: float) -> str:
    """Calculate what percentage of a given value is. E.g. 15% of 200."""
    result = (percentage / 100) * value
    return f"{percentage}% of {value} = {result}"

@tool
def count_words(text: str) -> str:
    """Counts the number of words in a given text string."""
    return f"The text contains {len(text.split())} words."

tools = [calculate_percentage, count_words]

# Bind tools directly to LLM — no AgentExecutor needed
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "What is 18% of 4500? And then explain what GST is in one sentence."
)

# Execute any tool calls the model made
import json
if response.tool_calls:
    print("📌 Agent called tools:")
    for tc in response.tool_calls:
        print(f"  Tool: {tc['name']}, Args: {tc['args']}")
        if tc["name"] == "calculate_percentage":
            print("  Result:", calculate_percentage.invoke(tc["args"]))
        elif tc["name"] == "count_words":
            print("  Result:", count_words.invoke(tc["args"]))

if response.content:
    print("\n📌 Final answer:", response.content)

📌 Agent called tools:
  Tool: calculate_percentage, Args: {'percentage': 18, 'value': 4500}
  Result: 18.0% of 4500.0 = 810.0


---
## Component 7: Document Loaders

Document loaders read external files and convert them into LangChain `Document` objects
ready for splitting, embedding, and retrieval.

You don't need to paste your entire knowledge base into every prompt —
load it once, embed it, retrieve only the relevant parts.

In [19]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a small knowledge base file about LangChain
knowledge_base = """
LangChain is an open-source framework for building LLM-powered applications.
It provides modular components: models, prompts, chains, memory, agents, tools, document loaders, and vector stores.

PromptTemplate separates prompt logic from application code, making prompts reusable and testable.
ChatPromptTemplate handles role-based conversations with system and human messages.

Chains connect components into pipelines using the LCEL pipe operator.
Simple chain pattern: prompt | model | output_parser

Memory stores conversation history between turns so the LLM appears stateful.
ConversationBufferMemory stores all messages. ConversationSummaryMemory compresses older turns.

Agents use a ReAct loop to decide which tools to call based on the user's request.
Tools are Python functions wrapped with the @tool decorator.

Document loaders ingest PDFs, web pages, text files, and more as Document objects.
Vector stores embed documents and enable semantic similarity search.
FAISS and ChromaDB are common local vector store choices.

RAG (Retrieval-Augmented Generation) combines vector retrieval with LLM generation.
Pipeline: user query → retrieve relevant chunks → inject as context → LLM generates grounded answer.

LangChain supports OpenAI, Anthropic, Groq, HuggingFace, and many other model providers.
LangGraph extends LangChain for stateful multi-agent workflows with graph-based control flow.
"""

# Save to file
with open("langchain_knowledge_base.txt", "w") as f:
    f.write(knowledge_base)

# Load using TextLoader
loader = TextLoader("langchain_knowledge_base.txt")
docs = loader.load()

# Split into chunks for embedding
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(f"📌 Document Loader output:")
print(f"  Loaded {len(docs)} document(s)")
print(f"  Split into {len(chunks)} chunks")
print(f"\nFirst chunk preview:")
print(chunks[0].page_content)

📌 Document Loader output:
  Loaded 1 document(s)
  Split into 7 chunks

First chunk preview:
LangChain is an open-source framework for building LLM-powered applications.
It provides modular components: models, prompts, chains, memory, agents, tools, document loaders, and vector stores.


---
## Component 8: Vector Stores + RAG Pipeline

Vector stores embed document chunks into numerical vectors and retrieve semantically similar ones.
This is the backbone of RAG — instead of keyword matching, you search by meaning.

We use **HuggingFace sentence-transformers** (completely free, runs locally)
and **ChromaDB** (local vector store, no external service needed).

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Free local embeddings — downloads once, runs offline after
print("Loading embedding model (downloads once ~90MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build the vector store from our chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="langchain-blog-demo"
)

# Test semantic search — finds by meaning, not just keywords
query = "How does memory work in LangChain?"
results = vectorstore.similarity_search(query, k=2)

print(f"\n📌 Semantic search for: '{query}'")
print("-" * 55)
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(doc.page_content)
    print()

Loading embedding model (downloads once ~90MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


📌 Semantic search for: 'How does memory work in LangChain?'
-------------------------------------------------------
Result 1:
LangChain is an open-source framework for building LLM-powered applications.
It provides modular components: models, prompts, chains, memory, agents, tools, document loaders, and vector stores.

Result 2:
Chains connect components into pipelines using the LCEL pipe operator.
Simple chain pattern: prompt | model | output_parser

Memory stores conversation history between turns so the LLM appears stateful.
ConversationBufferMemory stores all messages. ConversationSummaryMemory compresses older turns.



In [21]:
from langchain_core.runnables import RunnablePassthrough

# Build RAG chain: retrieve relevant context → inject into prompt → LLM answers
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer based only on the context provided.
If the answer is not in the context, say you don't know — do not make things up."""),
    ("human", """Context:\n{context}\n\nQuestion: {question}""")
])

# Full RAG pipeline using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test the RAG chain with questions about our knowledge base
rag_questions = [
    "What is the difference between ConversationBufferMemory and ConversationSummaryMemory?",
    "What is RAG and how does it work?",
    "What is LangGraph?"
]

print("📌 RAG Pipeline outputs — answers grounded in our knowledge base:")
print("=" * 60)
for q in rag_questions:
    print(f"\nQ: {q}")
    print("A:", rag_chain.invoke(q))
    print("-" * 40)

📌 RAG Pipeline outputs — answers grounded in our knowledge base:

Q: What is the difference between ConversationBufferMemory and ConversationSummaryMemory?
A: ConversationBufferMemory stores all messages, whereas ConversationSummaryMemory compresses older turns.
----------------------------------------

Q: What is RAG and how does it work?
A: RAG (Retrieval-Augmented Generation) is a pipeline that combines vector retrieval with Large Language Model (LLM) generation. It works as follows:

1. User query: The user submits a query.
2. Retrieve relevant chunks: The system retrieves relevant chunks or passages from a large corpus based on the user's query.
3. Inject as context: The retrieved chunks are injected as context into the LLM.
4. LLM generates grounded answer: The LLM generates an answer based on the injected context, providing a grounded response to the user's query.
----------------------------------------

Q: What is LangGraph?
A: LangGraph extends LangChain for stateful multi-ag

---
## Summary — Full Pipeline

```
User Input
    ↓
Prompt Template      ← structures the request
    ↓
LLM (Groq / Llama)   ← generates response
    ↓
Chain (LCEL)         ← orchestrates steps
    ↓                ↘
Output          Agent → Tools → result fed back
                       ↑
               Vector Store (RAG) + Memory
```

### Components used in this notebook
| # | Component | Class used | Purpose |
|---|---|---|---|
| 1 | LLM | `ChatGroq` | Core language model |
| 2 | Prompts | `PromptTemplate`, `ChatPromptTemplate` | Structure inputs |
| 3 | Chains | LCEL `|` operator | Compose pipeline steps |
| 4 | Memory | `ConversationBufferMemory` | Multi-turn context |
| 5 | Tools | `@tool` decorator | Expose functions to agent |
| 6 | Agents | `create_react_agent` | Dynamic tool selection |
| 7 | Document Loaders | `TextLoader` | Ingest external files |
| 8 | Vector Stores + RAG | `Chroma`, `HuggingFaceEmbeddings` | Semantic retrieval |

---

*Medium Blog: [https://medium.com/p/d079b6bbf2a4?postPublishedType=initial]*

*GitHub: https://github.com/AbhishekKharat04/Innomatics-Agentic-AI-Internship-2026/tree/main/GenAI%20–%20Prompt*